# Does this experiment reproduce?

A run produced a number. You run it again and get a different number. **Which changed:**
the code, the config, the data, the seed, the hardware — or nothing you recorded?

run-ledger answers that by recording a run's *identity* separately from its *outcome*.
Identity fields are hashed into a **fingerprint**; outcome fields are not. Which makes one
specific situation detectable:

> **same fingerprint + different metrics = something real is going unrecorded.**

This notebook produces that situation on purpose, then finds it.

---

**Before you run this:**

1. A ledger must be listening. From the repo root: `make build && ./bin/runledger &`
2. **You must be inside a git checkout.** `Run.start()` captures the commit before your
   training loop starts and refuses to proceed without one — a run whose code cannot be
   identified is not lineage. Colab and other hosted environments without a checkout will
   not work here.
3. `pip install -e ./python`

In [ ]:
import os, uuid
import runledger

SERVER = os.environ.get('RUNLEDGER_ADDR', 'http://localhost:8080')

# A fresh project name per execution, so re-running this notebook against a
# persistent ledger doesn't pile new runs onto the previous pass's groups.
PROJECT = f'notebook-demo-{uuid.uuid4().hex[:6]}'

# Fails loudly if nothing is listening -- reads raise rather than quietly
# returning an empty list, which would look like 'you have no runs'.
runledger.runs(server=SERVER, limit=1)
print(f'ledger at {SERVER} is up; recording into project {PROJECT!r}')

## A training step that isn't quite deterministic

`train()` below has a bug of the most ordinary kind: it uses the global RNG without
seeding it, so the `seed` we so carefully record has no effect on the result. This is
standing in for an unpinned dependency, a nondeterministic GPU kernel, or a data race —
the ledger cannot tell those apart, and that is the point.

In [ ]:
import random

def train(seed: int, lr: float) -> float:
    # The seed is recorded as identity but never actually used to seed the RNG.
    # That is the bug this notebook exists to surface.
    base = 0.5 - lr * 10
    return round(base + random.uniform(-0.05, 0.05), 4)

## Record the same experiment three times

Same project, same seed, same params, same commit. Identical identity — so all three
runs must land on the same fingerprint, whatever they measure.

In [ ]:
for i in range(3):
    with runledger.Run.start(
        project=PROJECT,
        seed=1,
        params={'lr': 0.003},
        # Passing a config_hash keeps this working even if your tree is dirty;
        # without one, a dirty tree is refused (ADR 0003).
        config_hash='notebook-cfg-v1',
        server=SERVER,
    ) as run:
        loss = train(seed=1, lr=0.003)
        run.log_metric('loss', loss)
    print(f'run {i}: loss={loss}  fingerprint={run.fingerprint[:16]}...')

Three runs, one fingerprint, three different losses. Read them back:

In [ ]:
rows = runledger.runs(project=PROJECT, server=SERVER)

for r in rows:
    print(f"{r['run_id']:<32} fp={r['fingerprint'][:16]}  loss={r['metrics']['loss']}")

print()
print('distinct fingerprints:', len({r['fingerprint'] for r in rows}))
print('distinct losses:      ', len({r['metrics']['loss'] for r in rows}))

## The verdict

`spread()` groups every run sharing a fingerprint and reports how far the metrics moved
across that group. One fingerprint, three repeats, and a standard deviation that is not zero:

In [ ]:
group, = runledger.spread(project=PROJECT, server=SERVER)

print('fingerprint:', group['fingerprint'][:16] + '...')
print('runs in group:', group['count'])
print()
for name, stat in group['metrics'].items():
    print(f"{name}: mean={stat['mean']:.4f}  stddev={stat['stddev']:.4f} "
          f"min={stat['min']}  max={stat['max']}")

if group.get('provenance'):
    print()
    print('provenance fields these runs disagree on:')
    for d in group['provenance']:
        print(' ', d['field'], '=', d['values'])

That `stddev` is the number that matters. It is this experiment's **reproducibility floor**:
any later 'improvement' smaller than this spread is indistinguishable from noise the
experiment already had.

The ledger cannot tell you *why* the runs differ — only that the explanation is not in the
record. That is the point at which you go looking.

In [ ]:
import matplotlib.pyplot as plt

losses = [r['metrics']['loss'] for r in rows]
stat = group['metrics']['loss']

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.scatter(range(len(losses)), losses, s=70, zorder=3, label='individual runs')
ax.axhline(stat['mean'], linestyle='--', linewidth=1, label=f"mean {stat['mean']:.4f}")
ax.axhspan(stat['mean'] - stat['stddev'], stat['mean'] + stat['stddev'],
           alpha=0.15, label=f"±1 stddev ({stat['stddev']:.4f})")
ax.set_xticks(range(len(losses)))
ax.set_xlabel('repeat')
ax.set_ylabel('loss')
ax.set_title('Same experiment, three times')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## Now change something that *is* recorded

Change the seed and the verdict has to change with it. A different seed is a different
experiment, so these runs get their own fingerprint — and differing metrics across two
*different* experiments is expected, not a finding.

In [ ]:
with runledger.Run.start(
    project=PROJECT,
    seed=2,                       # <- the only change
    params={'lr': 0.003},
    config_hash='notebook-cfg-v1',
    server=SERVER,
) as run:
    run.log_metric('loss', train(seed=2, lr=0.003))

print('new fingerprint:', run.fingerprint[:16] + '...')
print('matches the first group?', run.fingerprint == group['fingerprint'])

The ledger now holds two distinct experiments. `spread()` reports only the one with
repeats — a lone run has nothing to be inconsistent with, so it is not ranked:

In [ ]:
groups = runledger.spread(project=PROJECT, server=SERVER)
print(f'{len(groups)} group(s) with repeats, out of',
      len({r["fingerprint"] for r in runledger.runs(project=PROJECT, server=SERVER)}),
      'distinct experiments')

# Asking for the lone run's fingerprint directly still answers -- honestly.
lone, = runledger.spread(fingerprint=run.fingerprint, server=SERVER)
print('lone run reported as no_repeats:', lone['no_repeats'])

`no_repeats: True` rather than a standard deviation of zero. A single run has not
demonstrated reproducibility — it has simply never been asked to.

---

## What to take away

| | |
|---|---|
| Same fingerprint, same metrics | The experiment reproduces. |
| Same fingerprint, **different** metrics | Something affecting the result is not in the record. Go looking. |
| Different fingerprints, different metrics | Expected. You changed the experiment. |

Across a real project, `runledger.spread(project=...)` ranks every repeated experiment
worst-first, so the question *"which of my results should I not trust?"* has an answer
you can look up instead of remember.